# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

In [1]:
# imports

from ast import Mod
import os
import sys
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR

sys.path.append(os.path.abspath(os.path.join(os.getcwd(),'..')))
from ai_tools.tools import LLMQuery


In [2]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
if LITE_MODE:
    username = "Rodan009"
else:
    username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


# Before we look at the Artificial Neural Networks

## There is a different kind of Neural Network we could consider

In [4]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [5]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [6]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [7]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


Human predicted 120.0 for an item that actually costs 219.0


In [8]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

# And now - a vanilla Neural Network

During the remainder of this course we will get deeper into how Neural Networks work, and how to train a neural network.

This is just a sneak preview - let's make our own Neural Network, from scratch, using Pytorch.

Use this to get intuition; it's not important to know all about Neural networks at this point..

In [9]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [10]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [11]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# 1. Data Preparation
# Convert sparse matrix X to a dense numpy array because PyTorch layers expect dense inputs.
# Then convert to FloatTensor (32-bit float), which is the standard data type for neural network weights and calculations.
X_train_tensor = torch.FloatTensor(X.toarray())

# Convert labels to FloatTensor.
# .unsqueeze(1) reshapes the vector from [N] to [N, 1] (a column vector).
# This is crucial because loss functions (like BCELoss) differ mathematically between a flat vector and a column vector.
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# 2. Validation Split
# Split the dataset: 99% for training, 1% for validation.
# The validation set (X_val, y_val) acts as a proxy for "unseen data" during training.
# It helps us detect overfitting (when the model memorizes training data but fails on new data).
X_train, X_val, y_train, y_val = train_test_split(
    X_train_tensor, 
    y_train_tensor, 
    test_size=0.01, 
    random_state=42  # strict seed ensures we get the exact same split every time we run this
)

# 3. Data Loader
# TensorDataset wraps the tensors so we can access samples like tuples: (X_sample, y_sample).
train_dataset = TensorDataset(X_train, y_train)

# DataLoader handles the efficient feeding of data to the network.
# batch_size=64: The network learns from 64 examples at a time, not the whole dataset at once.
# This makes training faster and uses less memory (Mini-batch Gradient Descent).
# shuffle=True: Randomizes the order of samples every epoch. This breaks correlations in the data order,
# preventing the model from learning spurious patterns (like time-dependency) and helping it converge closer to the global minimum.
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# 4. Model Initialization
# Determine the number of input neurons based on the number of features (columns) in our data.
input_size = X_train_tensor.shape[1]

# Instantiate the Neural Network.
# This allocates memory for the model path and initializes the weights (usually with small random numbers).
# The model is now ready to receive data and start the forward-pass -> loss -> backward-pass cycle.
model = NeuralNetwork(input_size)

In [13]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [14]:
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# 1. Setup Training Components
# Define the "Loss Function" (or cost function). This mathematically measures how wrong the model's predictions are.
# MSELoss (Mean Squared Error) calculates the average squared difference between predictions and actual targets.
# Note: For binary classification, BCELoss is common, but MSE is also used sometimes for regression-like outputs.
loss_function = nn.MSELoss()

# Define the "Optimizer". This applies the math to update the model's weights based on the gradients.
# Adam is a popular adaptive algorithm (better than standard SGD).
# model.parameters() tells it which weights to update.
# lr=0.001 (Learning Rate) controls the size of the steps the optimizer takes. Too big = unstable, too small = slow.
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 2. Training Loop Configuration
# One "Epoch" is one complete pass through the entire training dataset.
# We will show the model the full dataset 2 times.
EPOCHS = 2

for epoch in range(EPOCHS):
    # Set the model to "Training Mode".
    # This is critical because some layers (like Dropout or BatchNorm) behave differently during training vs inference.
    model.train()
    
    # Iterate through the data in batches (e.g., 64 items at a time).
    # tqdm provides a progress bar.
    for batch_X, batch_y in tqdm(train_loader):
        
        # CLEAR GRADIENTS: PyTorch accumulates gradients by default.
        # We must zero them out before each new batch, otherwise we mix gradients from previous batches.
        optimizer.zero_grad()

        # --- The 4 Steps of Learning ---
        
        # 1. Forward Pass: Pass input data through the network layers to get predictions.
        outputs = model(batch_X)
        
        # 2. Compute Loss: Compare predictions (outputs) with ground truth (batch_y) to convert "wrongness" to a single number.
        loss = loss_function(outputs, batch_y)
        
        # 3. Backward Pass (Backpropagation): Calculate the gradient of the loss with respect to every weight in the network.
        # This tells us which direction to nudge each weight to reduce the error.
        loss.backward()
        
        # 4. Optimization Step: Actually update the weights using the gradients calculated above.
        optimizer.step()

    # 3. Validation Phase
    # Set the model to "Evaluation Mode". 
    # This freezes specific layers (like Dropout) ensuring reproducible and correct testing results.
    model.eval()
    
    # "No Grad" Context Manager: Tells PyTorch not to track gradients for operations inside this block.
    # Since we aren't training (updating weights) here, we save massive amounts of memory and computation.
    with torch.no_grad():
        # Run the validation set through the model to see how well it generalizes to unseen data.
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    # Log the status at the end of the epoch.
    # .item() converts a single-value Tensor (on GPU/CPU) to a standard Python number.
    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

100%|██████████| 310/310 [00:05<00:00, 57.57it/s]


Epoch [1/2], Train Loss: 16204.476, Val Loss: 20987.945


100%|██████████| 310/310 [00:04<00:00, 76.34it/s]

Epoch [2/2], Train Loss: 2241.602, Val Loss: 20880.561


### How PyTorch Loss and Optimization Work Under the Hood

The connection between the `loss` and the `optimizer` happens via a "shared blackboard" mechanism known as the **Computational Graph**. They never communicate directly; instead, they communicate through the Model's parameters (weights).

#### 1. The `loss` "knows" about the Model via the Graph
The variable `loss` is **not just a number**; it is a Tensor with a history (a computational graph) attached to it.

*   When you calculate `outputs = model(batch_X)`, PyTorch builds a **computational graph**. Every tensor in `outputs` remembers exactly which weights and mathematical operations created it.
*   When you calculate `loss = loss_function(outputs, batch_y)`, the `loss` tensor extends that graph. It knows *"I was created by `outputs`, and `outputs` was created by `model.layers[0].weight`."*

#### 2. `loss.backward()` Populates the Gradients
This is the bridge component. When you run `loss.backward()`:

1.  PyTorch travels **backwards** through that graph (from `loss` -> `outputs` -> `model weights`).
2.  It calculates the gradient (the error/slope) for each specific weight.
3.  **Crucially:** It stores that gradient **directly inside the parameter object itself** in an attribute called `.grad`.

Example of what happens conceptually:
```python
# Before backward()
print(model.layer1.weight.grad) # usually None or 0

loss.backward() 

# After backward()
print(model.layer1.weight.grad) # Contains the calculated error (e.g., 0.041)
```

#### 3. The Optimizer Reads the Parameters
The connection is indirect. The optimizer was initialized with a list of the model's parameters:
```python
# We gave the optimizer a list of pointers to the model's weights
optimizer = optim.Adam(model.parameters(), lr=0.001)
```
When you run optimizer.step():

1. The optimizer loops through that list of parameters we gave it at the start.
2. It checks the .grad attribute of each parameter (which was just filled by loss.backward()).
3. It updates the weight according to the optimization rule (e.g., Adam or SGD): weight = weight - (learning_rate * weight.grad)

Summary of the Flow
1. Model creates the graph: inputs -> outputs
2. Loss calculates error: outputs -> error
3. Backward deposits the "instructions" (gradients) into the Model's weights.
4. Optimizer reads those "instructions" from the Model's weights and updates the weights.

In [15]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [16]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$90 $64 $15 $55 $42 $169 $97 $94 $1 $32 $628 $259 $39 $56 $34 $4 $17 $46 $48 $17 $67 $38 $20 $14 $240 $311 $283 $15 $50 $43 $71 $11 $28 $11 $47 $295 $27 $22 $158 $71 $143 $11 $3 $54 $123 $25 $50 $29 $7 $67 $43 $91 $298 $33 $42 $48 $54 $68 $5 $20 $136 $14 $38 $60 $514 $22 $17 $282 $7 $93 $4 $15 $137 $86 $1 $36 $63 $34 $15 $61 $63 $53 $1 $48 $4 $3 $44 $122 $30 $75 $12 $15 $21 $45 $23 $98 $3 $28 $94 $278 $9 $43 $5 $65 $95 $15 $27 $286 $17 $143 $25 $155 $178 $23 $112 $319 $74 $11 $33 $19 $16 $64 $57 $3 $98 $7 $11 $227 $32 $52 $98 $25 $21 $11 $45 $29 $76 $13 $12 $8 $33 $204 $76 $23 $180 $57 $46 $325 $50 $5 $12 $187 $4 $91 $92 $141 $45 $12 $117 $4 $117 $4 $4 $21 $653 $9 $241 $22 $6 $57 $38 $15 $234 $87 $4 $2 $13 $32 $60 $106 $431 $6 $34 $1 $25 $70 $74 $31 $55 $11 $22 $36 $23 $28 $44 $20 $80 $135 $8 $7 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [6]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [ ]:
print(test[0].summary)

In [ ]:
messages_for(test[0])

In [4]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [7]:
gpt_4__1_nano(test[0])

'$249'

In [ ]:
test[0].price

In [ ]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$31 $34 $25 $20 $20 $130 $94 $65 $6 $180 $363 $71 $10 $6 $1 $8 $71 $5 $40 $79 $16 $6 $35 $74 $132 $303 $95 $5 $251 $60 $30 $15 $10 $50 $5 $119 $90 $30 $36 $13 $50 $45 $27 $45 $70 $0 $27 $13 $65 $52 $20 $115 $125 $0 $297 $16 $8 $80 $52 $3 $86 $28 $51 $10 $571 $30 $90 $295 $25 $74 $7 $8 $130 $6 $34 $21 $126 $5 $2 $1 $30 $3 $5 $69 $12 $0 $32 $156 $35 $21 $3 $25 $0 $10 $2 $78 $4 $93 $130 $325 $50 $33 $7 $11 $49 $32 $10 $370 $14 $49 $15 $136 $29 $8 $54 $80 $15 $5 $194 $97 $19 $411 $50 $16 $50 $90 $10 $151 $59 $19 $59 $13 $5 $5 $85 $0 $55 $15 $78 $12 $6 $50 $30 $16 $44 $18 $25 $390 $35 $17 $3 $144 $2 $10 $6 $71 $21 $41 $0 $25 $11 $19 $28 $3 $590 $3 $752 $30 $0 $5 $10 $3 $480 $77 $57 $101 $3 $57 $24 $13 $546 $25 $250 $101 $100 $3 $83 $63 $30 $12 $10 $99 $25 $11 $50 $70 $30 $30 $21 $1 

: 

In [ ]:
def claude_opus_4_5(item):
    response = completion(model="anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(claude_opus_4_5, test)

In [ ]:
def gemini_3_pro_preview(item):
    response = completion(model="gemini/gemini-3-pro-preview", messages=messages_for(item), reasoning_effort='low')
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_3_pro_preview, test, size=50, workers=2)

In [ ]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_2__5_flash_lite, test)

In [ ]:

def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [ ]:
evaluate(grok_4__1_fast, test)

In [ ]:
# The function for gpt-5.1

def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='high', seed=42)
    return response.choices[0].message.content


In [ ]:
evaluate(gpt_5__1, test)

In [ ]:
SYSTEM_PROMPT = "Estimate the price of a product based on a description, you **must** always return a price, no explanation."

In [11]:
from ai_tools.tools import LLMQuery, ModelName
def generic_price_guesser(item : Item, model : ModelName):
    client = LLMQuery(system_prompt=SYSTEM_PROMPT, model = model)
    price = client.query(item.summary, use_history = False)
    return price


In [19]:
generic_price_guesser(test[5], model = 'gemini-3-flash-preview')


Response content is empty and no tool calls found response=ChatCompletion(id='xS5xaZbyAtmqkdUP6p-nuQU', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1769025221, model='gemini-3-flash-preview', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=0, prompt_tokens=111, total_tokens=627, completion_tokens_details=None, prompt_tokens_details=None)), retrying


'$64.99'

In [21]:
test[5].summary

'Title: UPC Replacement Battery Cartridge for APC UPS Systems\nCategory: Electronics\nBrand: UPC PARTS\nDescription: Precharged, sealed-lead-acid replacement battery cartridge designed for APC UPS systems, enabling immediate use with plug-and-play installation.\nDetails: Includes all required connectors, fuses, and metal enclosure with no assembly needed; maintenance-free, pre-charged, and hot-swappable for easy installation.'

In [ ]:
evaluate(lambda item: generic_price_guesser(item, 'gemini-3-pro-preview'), test)

  0%|          | 0/200 [00:00<?, ?it/s]

$10 $24 $5 $5 $10 $100 $94 $70 $13 $170 $182 $120 $5 $2 $44 $7 $1 $15 $29 $24 $31 $14 $5 $55 $82 $104 $205 $1 $121 $60 $5 $35 $10 $55 $5 $170 $5 $43 $16 $15 $120 $35 $5 $105 $20 $2 $2 $2 $80 $7 $18 $93 $25 $0 $27 $5 $2 $60 $210 $1 $107 $48 $15 $65 $149 $10 $10 $325 $20 $6 $19 $1 $65 $4 $15 $20 $4 $2 $6 $2 $10 $0 $13 $69 $14 $25 $103 $66 $10 $1 $2 $40 $2 $10 $0 $100 $7 $42 $33 $125 $38 $7 $8 $10 $0 $8 $13 $360 $2 $100 $25 $71 $9 $73 $34 $10 $5 $10 $24 $82 $7 $49 $10 $76 $0 $15 $6 $21 $83 $94 $84 $26 $10 $5 $65 $1 $54 $35 $1 $11 $1 $149 $5 $0 $28 $8 $10 $45 $134 $8 $3 $133 $17 $10 $17 $51 $8 $36 $55 $10 $10 $19 $49 $0 $60 $7 $602 $14 $3 $0 

In [18]:
evaluate(lambda item: generic_price_guesser(item, 'gemini-3-flash-preview'), test, workers = 10, size = 100)

  0%|          | 0/100 [00:00<?, ?it/s]

$10 Response content is empty and no tool calls found response=ChatCompletion(id='1i1xaeKnN_6FkdUP2NKo8AU', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1769024982, model='gemini-3-flash-preview', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=0, prompt_tokens=93, total_tokens=399, completion_tokens_details=None, prompt_tokens_details=None)), retrying
$49 Response content is empty and no tool calls found response=ChatCompletion(id='2i1xacWcJM-EkdUPtKX40As', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1769024986, model='gemini-3-flash-preview', object='chat.completion', service_tier=None,

In [15]:
evaluate(lambda item: generic_price_guesser(item, "openai/gpt-oss-20b"), test, size = 100, workers = 10)

  0%|          | 0/100 [00:00<?, ?it/s]

$80 $164 $15 $32 $45 $20 $109 $305 $49 $80 $533 $99 $45 $5 $9 $7 $31 $45 $40 $102 $6 $86 $40 $55 $254 $173 $15 $10 $151 $65 $25 $0 $270 $45 $80 $69 $47 $34 $26 $18 $80 $40 $16 $45 $160 $10 $18 $13 $0 $32 $15 $112 $205 $35 $2147 $39 $11 $38 $298 $3 $137 $2 $44 $60 $379 $89 $125 $195 $195 $114 $3 $28 $155 $10 $15 $11 $156 $15 $18 $9 $230 $3 $1 $54 $15 $49 $117 $56 $130 $9 $5 $53 $5 $85 $4 $78 $1 $109 $228 $300 

In [16]:
evaluate(lambda item: generic_price_guesser(item, "openai/gpt-oss-120b"), test, size = 100, workers = 10)

  0%|          | 0/100 [00:00<?, ?it/s]

$0 $4 $5 $30 $19 $101 $109 $54 $9 $95 $384 $100 $28 $4 $39 $11 $70 $29 $80 $60 $89 $65 $44 $24 $182 $254 $204 $5 $160 $65 $58 $5 $139 $32 $84 $80 $70 $8 $75 $24 $70 $45 $15 $75 $71 $5 $52 $23 $46 $83 $22 $116 $76 $10 $176 $45 $3 $15 $77 $11 $37 $38 $42 $31 $410 $5 $145 $316 $5 $183 $12 $28 $101 $16 $20 $9 $155 $5 $3 $6 $10 $4 $30 $74 $5 $11 $37 $43 $20 $17 $3 $0 $9 $7 $1 $98 $5 $57 $140 $246 

In [17]:
evaluate(lambda item: generic_price_guesser(item, "deepseek/deepseek-v3.2"), test, size = 100, workers = 10)

  0%|          | 0/100 [00:00<?, ?it/s]

$31 $79 $19 $20 $15 $180 $118 $95 $10 $50 $63 $121 $10 $11 $44 $17 $9 $10 $140 $74 $9 $36 $65 $25 $83 $253 $195 $0 $81 $55 $100 $20 $115 $35 $15 $429 $90 $6 $24 $3 $140 $40 $10 $95 $70 $5 $32 $16 $75 $2 $16 $110 $355 $0 $17 $36 $4 $10 $167 $3 $91 $28 $9 $20 $134 $40 $142 $320 $50 $14 $16 $13 $195 $0 $22 $18 $96 $10 $2 $0 $30 $9 $0 $74 $2 $10 $2 $61 $25 $11 $0 $50 $8 $25 $2 $129 $4 $92 $50 $495 

In [23]:
evaluate(lambda item: generic_price_guesser(item, "z-ai/glm-4.7"), test, size = 100, workers = 10)

  0%|          | 0/100 [00:00<?, ?it/s]

$0 $4 $15 $20 $20 $180 $69 $112 $14 $270 $228 $30 $5 $14 $39 $8 $1 $5 $140 $74 $16 $16 $5 $65 $97 $174 $166 $0 $181 $60 $15 $30 $40 $45 $10 $219 $30 $45 $64 $22 $85 $40 $10 $10 $20 $10 $32 $2 $85 $22 $22 $105 $101 $15 $67 $9 $6 $120 $3 $1 $101 $43 $71 $55 $366 $11 $120 $320 $15 $54 $17 $0 $10 $6 $25 $14 $14 $0 $3 $2 $40 $3 $10 $74 $7 $5 $38 $119 $10 $11 $7 $15 $5 $20 $2 $108 $5 $92 $30 $100 

In [22]:
evaluate(lambda item: generic_price_guesser(item, "x-ai/grok-4.1-fast"), test, size = 100, workers = 10)

  0%|          | 0/100 [00:00<?, ?it/s]

$0 $114 $20 $59 $30 $160 $84 $10 $7 $151 $486 $320 $10 $8 $19 $13 $60 $15 $139 $79 $36 $96 $5 $254 $163 $24 $206 $9 $350 $55 $15 $10 $139 $50 $65 $170 $70 $31 $5 $8 $60 $35 $20 $74 $60 $8 $22 $13 $55 $48 $20 $115 $126 $1 $36 $95 $10 $130 $148 $1 $56 $38 $32 $30 $166 $19 $40 $256 $95 $143 $14 $43 $99 $4 $25 $16 $26 $3 $4 $6 $10 $4 $5 $74 $3 $30 $107 $123 $50 $16 $4 $85 $10 $10 $1 $39 $1 $7 $39 $4 

In [24]:
evaluate(lambda item: generic_price_guesser(item, "gemini-flash-lite-latest"), test, size = 100, workers = 10)

  0%|          | 0/100 [00:00<?, ?it/s]

$40 $70 $15 $60 $20 $114 $54 $120 $0 $120 $263 $71 $60 $9 $33 $7 $51 $4 $280 $34 $39 $9 $10 $175 $102 $193 $45 $5 $191 $63 $15 $15 $30 $45 $15 $119 $16 $31 $14 $13 $85 $45 $20 $35 $70 $0 $58 $13 $70 $88 $22 $94 $200 $0 $77 $16 $12 $80 $17 $7 $116 $38 $51 $15 $71 $30 $145 $255 $9 $74 $9 $9 $70 $6 $25 $13 $126 $5 $3 $0 $30 $1 $5 $74 $8 $25 $168 $144 $25 $21 $3 $20 $9 $16 $4 $38 $1 $77 $20 $125 

In [25]:
evaluate(lambda item: generic_price_guesser(item, "nvidia/nemotron-3-nano-30b-a3b"), test, size = 100, workers = 10)

  0%|          | 0/100 [00:00<?, ?it/s]

$10 $27 $5 $35 $15 $31 $104 $36 $13 $10 $113 $80 $26 $19 $9 $8 $51 $5 $240 $74 $89 $56 $45 $75 $133 $254 $54 $0 $200 $60 $65 $20 $90 $36 $30 $431 $25 $31 $15 $13 $135 $42 $25 $75 $71 $12 $27 $8 $85 $82 $29 $110 $275 $29 $297 $104 $2 $5 $67 $3 $36 $38 $56 $55 $626 $29 $155 $146 $125 $373 $117 $11 $95 $9 $35 $11 $225 $3 $2 $4 $40 $12 $0 $74 $14 $25 $3 $87 $79 $23 $2 $104 $5 $20 $1 $108 $25 $23 $71 $125 